# SETUP

In [1]:
from config import *
from components.vector_store import get_vector_store, delete_vector_store

In [2]:
CORPUS_CSV_PATH = "./dataset/corpus.csv" 
EVALUATION_CSV_PATH = "./dataset/evaluation.csv"
HUMAN_CURATED_CHUNKS_CSV_PATH = "./dataset/human_curated_chunks.csv"

### Đọc dataset

In [3]:
EXPERIMENTS = []
for method in EXPERIMENT_CHUNK_METHODS:
        for retrieval_mode in EXPERIMENT_RETRIEVAL_MODE:
            if method["method"] == "human_chunks" and retrieval_mode != "hybrid":
                continue
            experiment = {
                "method": method,
                "retrieval_mode": retrieval_mode,
            }
            EXPERIMENTS.append(experiment)                    

### Setup experiment configurations

In [4]:
def dict_value_std(dict):
    string = ""
    for key, value in dict.items():
        v = str(value).replace(":", "_")
        
        string += f"{key}_{v}"
        if key != list(dict.keys())[-1]:
            string += "_"
    return string

In [5]:
for exp in EXPERIMENTS:
    method_str = dict_value_std(exp["method"])
    collection_name = f"{method_str}_{exp['retrieval_mode'].value}"
    exp["collection_name"] = collection_name
    

### Xóa các collection trước đó
- Team mình chỉ chạy cell này khi thật sự cần thiết nha, tốn tiền!

In [6]:
# for exp in EXPERIMENTS:
#     delete_vector_store(exp["collection_name"])

### Tạo collection name (Nếu chưa tạo)

In [7]:
for exp in EXPERIMENTS:
    vector_store = get_vector_store(
        mode=exp["retrieval_mode"],
        collection_name=exp["collection_name"],
    ) 
    
    exp["vector_store"] = vector_store  

### Ingest vào vector store

In [8]:
import pandas as pd

In [9]:
corpus_df = pd.read_csv(CORPUS_CSV_PATH)
evaluation_df = pd.read_csv(EVALUATION_CSV_PATH)

In [10]:
corpus_df.head()

,id,user_id,name,full_text,document_metadata
0,10,02b3112c-e6da-4304-a148-9d53ce33a283,THÔNG TIN TUYỂN SINH HỆ ĐÀO TẠO VỪA LÀM VỪA HỌ...,Phương thức tuyển sinh - Đại học Vừa làm vừa h...,"{""topic"": ""Chung"", ""subtopic"": ""Chung""}"
1,11,02b3112c-e6da-4304-a148-9d53ce33a283,THÔNG TIN TUYỂN SINH SAU ĐẠI HỌC TRƯỜNG ĐẠI HỌ...,PHƯƠNG THỨC TUYỂN SINH CAO HỌC - Tuyển sinh cá...,"{""topic"": ""Chung"", ""subtopic"": ""Chung""}"
2,6,02b3112c-e6da-4304-a148-9d53ce33a283,Giới thiệu Trường Đại học Công nghệ Kỹ thuật TP. Hồ Chí Minh,Phần giới thiệu Trường Đại học Sư phạm Kỹ thuậ...,"{""topic"": ""Chung"", ""subtopic"": ""Chung""}"
3,8,02b3112c-e6da-4304-a148-9d53ce33a283,Thông tin tuyển sinh Đại học Chính quy các ngà...,"Ngành Kiến trúc Trang bị kiến thức, kỹ năng về...","{""topic"": ""Chung"", ""subtopic"": ""Chung""}"
4,9,02b3112c-e6da-4304-a148-9d53ce33a283,Thông tin về tuyển sinh Đại học Hệ liên kết đà...,Đối tượng tuyển sinh Học sinh của tất cả c...,"{""topic"": ""Chung"", ""subtopic"": ""Chung""}"


In [11]:
evaluation_df.head()

,id,question,answer,relevant_doc_ids
0,1,Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Min...,Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Min...,NaN
1,2,Vị trí Trường Đại học Sư phạm Kỹ thuật TP. Hồ ...,Tọa lạc tại Thành phố Thủ Đức – khu vực cửa ng...,NaN
2,3,Ưu thế của trường Đại học Sư phạm Kỹ thuật TP....,Một trong những điểm nổi bật trong chương trìn...,NaN
3,4,Hãy nêu cho em biết 1 vài lý do để em mạnh dạn...,Trường Công lập với bề dày lịch sử trên 60 năm...,NaN
4,5,HCMUTE có những ngành đào tạo nào trong năm 2025?,"Năm 2025, Trường Đại học Sư phạm Kỹ thuật TP. ...",NaN


### EVALUATE: RETRIEVAL PERFORMANCE

#### Corpus load

In [12]:
# import json
# from langchain_core.documents import Document
# documents = []
# for _, row in corpus_df.iterrows():
#     document_metadata = json.loads(row["document_metadata"]),
#     doc = Document(
#         page_content=row["name"] + "\n"  + row["full_text"],
#         metadata={
#             "document_id": row["id"],
#             "additional_metadata": document_metadata
#         }
#     )
#     documents.append(doc)
    
# print(f"Total documents: {len(documents)}")

### Curated chunks load

In [13]:
# human_curated_chunks_df = pd.read_csv(HUMAN_CURATED_CHUNKS_CSV_PATH)
# human_curated_chunks_df = human_curated_chunks_df.merge(
#     corpus_df[["id", "name", "document_metadata"]].rename(columns={"id": "document_id"}),
#     on="document_id",
#     how="left"
# )


In [14]:
# import json
# from langchain_core.documents import Document
# human_curated_chunk_documents = []
# for _, row in human_curated_chunks_df.iterrows():
#     document_metadata = json.loads(row["document_metadata"]),
#     doc = Document(
#         page_content=row["name"] + "\n"  + row["text"],
#         metadata={
#             "document_id": row["id"],
#             "additional_metadata": document_metadata
#         }
#     )
#     human_curated_chunk_documents.append(doc)
    
# print(f"Total human curated chunk documents: {len(human_curated_chunk_documents)}")

In [15]:
# import asyncio
# from typing import Dict, Tuple

# from components.embeddings import get_dense_embedding_model
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_experimental.text_splitter import SemanticChunker
# from components.chunker import LLMTextSplitter

# chunk_cache: Dict[Tuple, list] = {}


# def make_chunk_key(method_cfg: dict) -> Tuple:
#     return tuple(sorted(method_cfg.items()))


# async def run_chunking():
#     for exp in EXPERIMENTS:
#         method_cfg = exp["method"]
#         retrieval_mode = exp["retrieval_mode"]

#         chunk_key = make_chunk_key(method_cfg)

#         if chunk_key in chunk_cache:
#             exp["chunked_documents"] = chunk_cache[chunk_key]
#             print(
#                 f"[CACHE HIT] method={method_cfg} | "
#                 f"retrieval_mode={retrieval_mode} | "
#                 f"chunks={len(chunk_cache[chunk_key])}"
#             )
#             continue

#         method = method_cfg["method"]

#         if method == "human_chunks":
#             chunked_documents = human_curated_chunk_documents

#         elif method == "naive_chunks":
#             chunker = RecursiveCharacterTextSplitter(
#                 chunk_size=method_cfg["chunk_size"],
#                 chunk_overlap=method_cfg["chunk_overlap"],
#             )
#             chunked_documents = chunker.split_documents(documents)

#         elif method == "sem_chunks":
#             chunker = SemanticChunker(
#                 breakpoint_threshold_amount=method_cfg["breakpoint_threshold_amount"],
#                 min_chunk_size=method_cfg["min_chunk_size"],
#                 embeddings=get_dense_embedding_model(),
#             )
#             chunked_documents = chunker.split_documents(documents)

#         elif method == "llm_chunks":
#             chunker = LLMTextSplitter(
#                 model_name=method_cfg.get("model_name"),
#             )

#             chunked_documents = await chunker.asplit_documents(documents)

#         else:
#             raise ValueError(f"Unknown chunking method: {method}")
#         chunk_cache[chunk_key] = chunked_documents
#         exp["chunked_documents"] = chunked_documents

#         print(
#             f"[CHUNKED] method={method_cfg} | "
#             f"chunks={len(chunked_documents)}"
#         )


# await run_chunking()


### Ingest

In [16]:
# for exp in EXPERIMENTS:
#     vector_store = exp["vector_store"]
#     documents = exp["chunked_documents"]
#     vector_store.add_documents(documents)

### Retrieve experiments

In [17]:
from config import TEXT2SQL_DOC_ID
from components.pipelines.our_rag import OurRAGPipeline, load_table_schema, execute_sql_query
from components.prompt import TOOL_SELECTION_PROMPT
import asyncio
async def get_text2sql_context(question: str, vector_store) -> tuple:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage
    from config import settings
    
    schema = load_table_schema()
    llm = ChatOpenAI(model=EXPERIMENT_MODEL["model_name"], temperature=EXPERIMENT_MODEL["temperature"], api_key=settings.api_key)
    
    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'document_search_tool',
                'description': 'Tìm kiếm thông tin mô tả, hướng dẫn từ kho tài liệu HCMUTE. Sử dụng cho câu hỏi về: giới thiệu ngành học, chương trình đào tạo, quy chế tuyển sinh, thủ tục đăng ký, cơ sở vật chất, đời sống sinh viên, hướng dẫn và thông báo.',
                'parameters': {
                    'type': 'object',
                    'properties': {'query': {'type': 'string', 'description': 'Câu truy vấn tìm kiếm tài liệu'}},
                    'required': ['query']
                }
            }
        },
        {
            'type': 'function',
            'function': {
                'name': 'text2sql_tool',
                'description': 'Truy vấn số liệu cụ thể từ cơ sở dữ liệu HCMUTE. Sử dụng cho câu hỏi về: điểm chuẩn, chỉ tiêu tuyển sinh, học phí, mã ngành, tỷ lệ chọi, và các thông tin định lượng khác.',
                'parameters': {
                    'type': 'object',
                    'properties': {'query_text': {'type': 'string', 'description': 'Câu hỏi cần chuyển thành SQL'}},
                    'required': ['query_text']
                }
            }
        }
    ]
    
    prompt = TOOL_SELECTION_PROMPT.format(schema=schema, question=question)
    response = await llm.bind(tools=tools).ainvoke([HumanMessage(content=prompt)])
    
    use_text2sql = False
    sql_context = ''
    
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tool_call in response.tool_calls:
            if tool_call['name'] == 'text2sql_tool':
                use_text2sql = True
                sql_context = execute_sql_query(tool_call['args'].get('query_text', question))
                break
    
    return use_text2sql, sql_context


In [18]:
evaluation_df.columns

Index(['id', 'question', 'answer', 'relevant_doc_ids'], dtype='object')

In [19]:
top_k_values = [1, 3, 5, 10]
retrieval_eval_results = []
RETRIEVAL_EVAL_RESULT_OUTPUT_CSV_PATH = "./output/retrieval_eval_results.csv"

In [20]:
def build_top_k_context(
    *,
    all_vector_docs,
    top_k,
    use_text2sql,
    sql_context,
    text2sql_doc_id,
):
    """
    Returns:
      retrieved_doc_ids, context_parts
    Ensures:
      - length == top_k
      - SQL doc is LAST if used
    """
    if use_text2sql:
        vector_k = top_k - 1
        vector_docs = all_vector_docs[:vector_k]

        retrieved_doc_ids = [
            doc.metadata.get("document_id") for doc in vector_docs
        ] + [text2sql_doc_id]

        context_parts = (
            [doc.page_content for doc in vector_docs]
            + [sql_context]
        )
    else:
        vector_docs = all_vector_docs[:top_k]

        retrieved_doc_ids = [
            doc.metadata.get("document_id") for doc in vector_docs
        ]
        context_parts = [doc.page_content for doc in vector_docs]

    return retrieved_doc_ids, context_parts


In [21]:
# async def run_single_exp(exp):
#     results = []

#     print(f"Evaluating for vector store: {exp['collection_name']}")

#     vector_store = exp["vector_store"]
#     method_name = exp["method"]["method"]

#     max_top_k = max(top_k_values)

#     for row_idx, row in enumerate(evaluation_df.iterrows(), start=1):
#         _, row = row
#         question = row["question"]

#         print(f" Question {row_idx}/{len(evaluation_df)}")

#         # -------------------------
#         # Decide text2sql ONCE
#         # -------------------------
#         use_text2sql = False
#         sql_context = None

#         if method_name == "human_chunks":
#             use_text2sql, sql_context = await get_text2sql_context(
#                 question, vector_store
#             )

#         # -------------------------
#         # Compute max vector_k
#         # -------------------------
#         max_vector_k = (
#             max_top_k - 1
#             if method_name == "human_chunks" and use_text2sql
#             else max_top_k
#         )

#         print(
#             f"text2sql={'enabled' if use_text2sql else 'disabled'}, "
#             f"max_vector_k={max_vector_k}"
#         )

#         # -------------------------
#         # Retrieve ONCE
#         # -------------------------
#         all_vector_docs = vector_store.similarity_search(
#             query=question,
#             k=max_vector_k
#         )

#         # -------------------------
#         # Build each @k explicitly
#         # -------------------------
#         for i, top_k in enumerate(top_k_values, start=1):
#             print(f"  [{i}/{len(top_k_values)}] Evaluating top_k={top_k}")

#             retrieved_doc_ids, context_parts = build_top_k_context(
#                 all_vector_docs=all_vector_docs,
#                 top_k=top_k,
#                 use_text2sql=(method_name == "human_chunks" and use_text2sql),
#                 sql_context=sql_context,
#                 text2sql_doc_id=TEXT2SQL_DOC_ID,
#             )

#             assert len(retrieved_doc_ids) == top_k, (
#                 f"Expected {top_k}, got {len(retrieved_doc_ids)}"
#             )

#             # Debug safety check
#             if method_name == "human_chunks" and use_text2sql:
#                 assert retrieved_doc_ids[-1] == TEXT2SQL_DOC_ID

#             results.append({
#                 "question": question,
#                 "top_k": top_k,
#                 "retrieved_doc_ids": retrieved_doc_ids,
#                 "method": dict_value_std(exp["method"]).replace("method_", ""),
#                 "context": "\n".join(context_parts),
#                 **row.to_dict()
#             })

#             print("   done\n")

#     print(f"Finished: {exp['collection_name']}")
#     return results


In [22]:
# import asyncio

# async def run_retrieval_eval_parallel():
#     tasks = [
#         run_single_exp(exp)
#         for exp in EXPERIMENTS
#     ]

#     all_results = await asyncio.gather(*tasks)

#     flattened = [
#         item
#         for exp_results in all_results
#         for item in exp_results
#     ]

#     return flattened


In [23]:
# retrieval_eval_results = await run_retrieval_eval_parallel()

# retrieval_eval_df = pd.DataFrame(retrieval_eval_results)
# retrieval_eval_df.to_csv(
#     "./output/retrieval_eval_results.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

In [24]:
# retrieval_result_df = pd.read_csv(RETRIEVAL_EVAL_RESULT_OUTPUT_CSV_PATH)

In [25]:
# retrieval_result_df["method"].unique()

### EVALUATE: QA PERFORMANCE

In [26]:
import asyncio
import pandas as pd
from typing import List, Dict
from tqdm import tqdm
import json
import os
from pathlib import Path

from config import settings, EXPERIMENT_RERANKER, LLM_ONLY_PIPELINE_MODEL_NAME
from components.vector_store import get_vector_store
from components.reranks import JinaReranker
from components.pipelines.our_rag import OurRAGPipeline
from components.pipelines.basic_rag import BasicRAGPipeline
from components.pipelines.self_rag import SelfRAGPipeline
from components.pipelines.llm_only import LLMOnlyPipeline
from langchain_qdrant import RetrievalMode


In [27]:
EVALUATION_CSV_PATH = "./dataset/evaluation.csv"
OUTPUT_CSV_PATH = "./output/qa_performance_results.csv"

FLUSH_EVERY = 100

VECTOR_STORE_CONFIGS = {
    "human_chunks": "method_human_chunks_hybrid",
    "llm_chunks": "method_llm_chunks_model_openai_gpt-5-mini_hybrid",
    "sem_chunks": "method_sem_chunks_breakpoint_threshold_amount_70_min_chunk_size_512_hybrid",
    "naive_chunks": "method_naive_chunks_chunk_size_512_chunk_overlap_64_hybrid"
}

PIPELINE_CONFIG = {
    "k": 10,
    "model_name": "gpt-5-mini",
    "temperature": 0.7,
    "rerank_top_k": 5
}


In [28]:
eval_df = pd.read_csv(EVALUATION_CSV_PATH)

vector_stores = {}
for method_name, collection_name in VECTOR_STORE_CONFIGS.items():
    try:
        vector_stores[method_name] = get_vector_store(
            mode=RetrievalMode.HYBRID,
            collection_name=collection_name
        )
        print(f"Loaded vector store: {method_name}")
    except Exception as e:
        print(f"Failed to load {method_name}: {e}")
        raise

csv_lock = asyncio.Lock()

Loaded vector store: human_chunks
Loaded vector store: llm_chunks
Loaded vector store: sem_chunks
Loaded vector store: naive_chunks


In [29]:
eval_df = pd.read_csv(EVALUATION_CSV_PATH)
eval_df.head()  

,id,question,answer,relevant_doc_ids
0,1,Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Min...,Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Min...,NaN
1,2,Vị trí Trường Đại học Sư phạm Kỹ thuật TP. Hồ ...,Tọa lạc tại Thành phố Thủ Đức – khu vực cửa ng...,NaN
2,3,Ưu thế của trường Đại học Sư phạm Kỹ thuật TP....,Một trong những điểm nổi bật trong chương trìn...,NaN
3,4,Hãy nêu cho em biết 1 vài lý do để em mạnh dạn...,Trường Công lập với bề dày lịch sử trên 60 năm...,NaN
4,5,HCMUTE có những ngành đào tạo nào trong năm 2025?,"Năm 2025, Trường Đại học Sư phạm Kỹ thuật TP. ...",NaN


In [30]:
vector_stores = {
    "dense": {},
    "hybrid": {}
}

for method_name, collection_name in VECTOR_STORE_CONFIGS.items():
    vector_stores["dense"][method_name] = get_vector_store(
        mode=RetrievalMode.DENSE,
        collection_name=collection_name
    )

    vector_stores["hybrid"][method_name] = get_vector_store(
        mode=RetrievalMode.HYBRID,
        collection_name=collection_name
    )

    print(f"Loaded dense + hybrid stores: {method_name}")


Loaded dense + hybrid stores: human_chunks
Loaded dense + hybrid stores: llm_chunks
Loaded dense + hybrid stores: sem_chunks
Loaded dense + hybrid stores: naive_chunks


In [31]:
csv_lock = asyncio.Lock()
buffer: List[Dict] = []
total_written = 0
header_written = False

In [32]:
async def append_result(result: Dict):
    async with csv_lock:
        buffer.append(result)

        if len(buffer) >= FLUSH_EVERY:
            await flush_buffer()

In [33]:
async def flush_buffer():
    global buffer, total_written, header_written

    if not buffer:
        return

    df = pd.DataFrame(buffer)

    write_header = not header_written

    df.to_csv(
        OUTPUT_CSV_PATH,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig"
    )

    total_written += len(buffer)
    buffer.clear()
    header_written = True

    print(f"Flushed {total_written} rows")

In [34]:
async def run_pipeline_config(config: Dict, pipelines: Dict, pbar: tqdm):
    method = config["method"]
    pipeline_name = config["pipeline"]
    pipeline_key = config["pipeline_key"]
    retrieval_mode = config["retrieval_mode"]

    pipeline = pipelines[pipeline_key]

    if method != "llm_only":
        pipeline.vector_store = vector_stores[retrieval_mode][method]

    for _, row in eval_df.iterrows():
        question = row["question"]

        try:
            result = await pipeline.run(question)

            result_dict = {
                "id": row["id"],
                "question": result.question,
                "ground_truth_doc_ids": (
                    row["relevant_doc_ids"]
                    if pd.notna(row["relevant_doc_ids"])
                    else None
                ),
                "ground_truth_answer": row["answer"],
                "generated_answer": result.answer,
                "method": method,
                "context": result.context,
                "retrieved_doc_ids": json.dumps(result.doc_ids or []),
                "pipeline": pipeline_name,
                "retrieval_mode": retrieval_mode
            }

        except Exception as e:
            result_dict = {
                "id": row["id"],
                "question": question,
                "ground_truth_doc_ids": (
                    row["relevant_doc_ids"]
                    if pd.notna(row["relevant_doc_ids"])
                    else None
                ),
                "ground_truth_answer": row["answer"],
                "generated_answer": f"ERROR: {str(e)}",
                "method": method,
                "context": "",
                "retrieved_doc_ids": "[]",
                "pipeline": pipeline_name,
                "retrieval_mode": retrieval_mode
            }

        await append_result(result_dict)
        pbar.update(1)


In [35]:
from pathlib import Path

In [36]:
async def run_qa_evaluation():
    Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
    if os.path.exists(OUTPUT_CSV_PATH):
        os.remove(OUTPUT_CSV_PATH)

    reranker = None
    if EXPERIMENT_RERANKER.get("enabled"):
        reranker = JinaReranker(
            model_id=EXPERIMENT_RERANKER["model_id"],
            api_key=settings.jina_api_key
        )

    pipelines = {
        "our_rag": OurRAGPipeline(
            vector_store=None,   
            k=PIPELINE_CONFIG["k"],
            model_name=PIPELINE_CONFIG["model_name"],
            temperature=PIPELINE_CONFIG["temperature"],
            reranker=None,
            use_query_expansion=True
        ),
        "our_rag_rerank": OurRAGPipeline(
            vector_store=None,  
            k=PIPELINE_CONFIG["k"],
            model_name=PIPELINE_CONFIG["model_name"],
            temperature=PIPELINE_CONFIG["temperature"],
            reranker=reranker,
            rerank_top_k=PIPELINE_CONFIG["rerank_top_k"],
            use_query_expansion=True
        ),
        "basic_rag": BasicRAGPipeline(
            vector_store=None,   
            k=PIPELINE_CONFIG["k"],
            model_name=PIPELINE_CONFIG["model_name"],
            temperature=PIPELINE_CONFIG["temperature"]
        ),
        "self_rag": SelfRAGPipeline(
            vector_store=None,
            k=PIPELINE_CONFIG["k"],
            model_name=PIPELINE_CONFIG["model_name"],
            temperature=PIPELINE_CONFIG["temperature"],
            max_retries=2
        ),
        "llm_only": LLMOnlyPipeline(
            model_name=LLM_ONLY_PIPELINE_MODEL_NAME,
            temperature=PIPELINE_CONFIG["temperature"]
        )
    }


    eval_configs = [
        {"method": "human_chunks", "pipeline": "our_rag", "pipeline_key": "our_rag", "retrieval_mode": "hybrid"},
        {"method": "llm_chunks", "pipeline": "our_rag_rerank", "pipeline_key": "our_rag_rerank", "retrieval_mode": "hybrid"},

        # BasicRAG (dense + hybrid)
        {"method": "llm_chunks", "pipeline": "basic_rag", "pipeline_key": "basic_rag", "retrieval_mode": "dense"},
        {"method": "llm_chunks", "pipeline": "basic_rag", "pipeline_key": "basic_rag", "retrieval_mode": "hybrid"},
        {"method": "sem_chunks", "pipeline": "basic_rag", "pipeline_key": "basic_rag", "retrieval_mode": "dense"},
        {"method": "sem_chunks", "pipeline": "basic_rag", "pipeline_key": "basic_rag", "retrieval_mode": "hybrid"},

        # SelfRAG (dense + hybrid)
        {"method": "llm_chunks", "pipeline": "self_rag", "pipeline_key": "self_rag", "retrieval_mode": "dense"},
        {"method": "llm_chunks", "pipeline": "self_rag", "pipeline_key": "self_rag", "retrieval_mode": "hybrid"},
        {"method": "sem_chunks", "pipeline": "self_rag", "pipeline_key": "self_rag", "retrieval_mode": "dense"},
        {"method": "sem_chunks", "pipeline": "self_rag", "pipeline_key": "self_rag", "retrieval_mode": "hybrid"},

        # LLM only
        {"method": "llm_only", "pipeline": "llm_only", "pipeline_key": "llm_only", "retrieval_mode": None},
    ]


    total_evals = len(eval_configs) * len(eval_df)
    pbar = tqdm(total=total_evals, desc="Progress")

    tasks = [
        run_pipeline_config(config, pipelines, pbar)
        for config in eval_configs
    ]

    await asyncio.gather(*tasks)

    async with csv_lock:
        await flush_buffer()

    pbar.close()
    print("Evaluation complete")


In [37]:
await run_qa_evaluation()

Progress:   6%|▌         | 99/1650 [02:48<39:44,  1.54s/it]  

Flushed 100 rows


Progress:  12%|█▏        | 200/1650 [05:36<42:41,  1.77s/it]  

Flushed 200 rows


Progress:  18%|█▊        | 300/1650 [08:22<44:24,  1.97s/it]  

...

Flushed 1500 rows


Progress:  97%|█████████▋| 1600/1650 [1:09:46<01:38,  1.97s/it]

Flushed 1600 rows


Progress: 100%|██████████| 1650/1650 [1:12:21<00:00,  2.63s/it]

Flushed 1650 rows
Evaluation complete
